# Reddit Kaggle Competition: Inference Notebook

This notebook loads a trained model and generates predictions on the test set for submission.

### The Goal
Predict the **probability** (0.0 to 1.0) for each of the 28 emotion columns for every test sample.

### The Metric
Submissions are evaluated using **Macro ROC AUC**.

---

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import re
import json
import os

# Configuration
DATA_DIR = '../data/'
MODEL_PATH = '/tmp/model.pt'
VOCAB_PATH = '/tmp/vocab.json'
BATCH_SIZE = 64
MAX_LEN = 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


## 1. Load the Data
We load the test CSV and recover the emotion column names from the training file.

In [2]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_kaggle.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test_kaggle.csv'))

# The emotion labels are all columns except ID and text
emotion_cols = [c for c in train_df.columns if c not in ['ID', 'text']]
num_labels = len(emotion_cols)

print(f"Test samples: {len(test_df)}")
print(f"Total emotions: {num_labels}")

test_df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: '/content/train_kaggle.csv'

## 2. Download Model Artifacts
Use `gdown` to pull the trained model weights and saved vocabulary from Google Drive.

In [ ]:
# !pip install gdown -q
# import gdown

# TODO: Replace with your actual Google Drive file IDs
# MODEL_FILE_ID = 'YOUR_MODEL_FILE_ID'
# VOCAB_FILE_ID  = 'YOUR_VOCAB_FILE_ID'

# gdown.download(f'https://drive.google.com/uc?id={MODEL_FILE_ID}', MODEL_PATH, quiet=False)
# gdown.download(f'https://drive.google.com/uc?id={VOCAB_FILE_ID}',  VOCAB_PATH,  quiet=False)

print("Artifacts ready.")

## 3. Text Preprocessing & Vocabulary
We restore the same tokenization and word-to-index mapping that was used during training.

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

# Load vocabulary saved during training
with open(VOCAB_PATH, 'r') as f:
    vocab = json.load(f)

vocab_size = len(vocab)

def text_to_ids(text):
    ids = [vocab.get(w, vocab.get('<UNK>', 1)) for w in tokenize(text)][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

print(f"Vocabulary size: {vocab_size}")

## 4. Dataset and DataLoader

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df):
        self.texts = df['text'].apply(text_to_ids).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long)

test_loader = DataLoader(EmotionDataset(test_df), batch_size=BATCH_SIZE)

## 5. Model Definition & Loading
Define the same architecture used in training, then load the saved weights.

In [ ]:
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, dropout_p=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout_p)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        # hidden: [2, batch, hidden_dim] — concat forward and backward final states
        hidden = torch.cat([hidden[0], hidden[1]], dim=1)
        return self.fc(self.dropout(hidden))

# TODO: Make sure these hyperparameters match the values used during training
model = BiLSTMModel(vocab_size, 100, 128, num_labels).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("Model loaded and ready for inference.")

## 6. Generate Submission
Run the model on the test set and save the probabilities.

In [ ]:
probs = []
with torch.no_grad():
    for x in test_loader:
        logits = model(x.to(DEVICE))
        probs.append(torch.sigmoid(logits).cpu().numpy())

submission = pd.DataFrame(np.vstack(probs), columns=emotion_cols)
submission.insert(0, 'ID', test_df['ID'].values)
submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")